# 05 — DistilBERT Fine-tuning

**Prérequis** : `01_clustering.ipynb` exécuté.

**Données** : `../data/client_text_data.pkl`, `../data/label_maps.json`

**Modèle** : `distilbert-base-uncased` (HuggingFace) — utilise son propre tokenizer, pas le vocab custom.

**Sortie** : `../models/distilbert_best.pt`, `../data/results_distilbert.pkl`

## Bloc 1 : Config & imports

In [ ]:
import os, pickle, json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR   = '../data/'
MODEL_DIR  = '../models/'
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_NAME   = 'distilbert-base-uncased'
N_CLASSES    = 8
MAX_LEN      = 128   # DistilBERT supporte jusqu'à 512 ; 128 suffit pour les tweets
BATCH_SIZE   = 32    # plus petit que RNN/LSTM car DistilBERT consomme plus de mémoire GPU
LR           = 2e-5  # fine-tuning : LR très faible pour ne pas détruire les poids pré-entraînés
N_EPOCHS     = 3     # 3 epochs suffisent pour le fine-tuning
RANDOM_STATE = 42

torch.manual_seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

## Bloc 2 : Chargement des données texte

In [ ]:
with open(DATA_DIR + 'client_text_data.pkl', 'rb') as f:
    data = pickle.load(f)   # {'train': [(text, label), ...], 'val': [...], 'test': [...]}
with open(DATA_DIR + 'label_maps.json') as f:
    label_maps = json.load(f)

CLIENT_LABELS = {int(k): v for k, v in label_maps['client'].items()}
label_names   = [CLIENT_LABELS[i] for i in range(N_CLASSES)]

print(f'Train : {len(data["train"]):,} | Val : {len(data["val"]):,} | Test : {len(data["test"]):,}')
print(f'Classes ({N_CLASSES}) : {label_names}')
print(f'Exemple : "{data["train"][0][0][:80]}" → label {data["train"][0][1]}')

## Bloc 3 — Tokenizer & Dataset

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
print(f'Tokenizer chargé : {MODEL_NAME} | vocab={tokenizer.vocab_size:,}')


class BertIntentDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.texts    = [d[0] for d in data]
        self.labels   = [d[1] for d in data]
        self.tokenizer = tokenizer
        self.max_len  = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(0),
            'attention_mask' : encoding['attention_mask'].squeeze(0),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long)
        }


loaders = {
    split: DataLoader(
        BertIntentDataset(data[split], tokenizer, MAX_LEN),
        batch_size=BATCH_SIZE,
        shuffle=(split == 'train')
    )
    for split in ('train', 'val', 'test')
}

batch = next(iter(loaders['train']))
print(f'Batch — input_ids: {batch["input_ids"].shape}, attention_mask: {batch["attention_mask"].shape}')

## Bloc 4 : Modèle DistilBERT

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=N_CLASSES
).to(device)

print(f'Paramètres totaux    : {sum(p.numel() for p in model.parameters()):,}')
print(f'Paramètres entraîn. : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Bloc 5 : Entraînement

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

train_losses, val_losses, val_accs = [], [], []
best_val_acc = 0.0

for epoch in range(N_EPOCHS):
    #  Train 
    model.train()
    total_loss = 0.0
    for batch in loaders['train']:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += outputs.loss.item()
    train_losses.append(total_loss / len(loaders['train']))

    #  Validation 
    model.eval()
    val_loss, preds, targets = 0.0, [], []
    with torch.no_grad():
        for batch in loaders['val']:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            preds.extend(torch.argmax(outputs.logits, dim=1).cpu().tolist())
            targets.extend(labels.cpu().tolist())

    val_losses.append(val_loss / len(loaders['val']))
    acc = accuracy_score(targets, preds)
    val_accs.append(acc)

    print(f'Epoch {epoch+1}/{N_EPOCHS} | train={train_losses[-1]:.4f} | val={val_losses[-1]:.4f} | val_acc={acc:.4f}', end='')
    if acc > best_val_acc:
        best_val_acc = acc
        torch.save(model.state_dict(), MODEL_DIR + 'distilbert_best.pt')
        print(' ✓')
    else:
        print()

## Bloc 6 : Courbes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, label='Train'); axes[0].plot(val_losses, label='Val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(val_accs, color='green')
axes[1].axhline(best_val_acc, color='red', linestyle='--', label=f'Best={best_val_acc:.4f}')
axes[1].set_title('Val Accuracy'); axes[1].legend()
plt.suptitle('DistilBERT — Courbes fine-tuning')
plt.tight_layout()
plt.savefig(MODEL_DIR + 'distilbert_curves.png', dpi=100)
plt.show()

## Bloc 7 : Évaluation sur le test set

In [ ]:
model.load_state_dict(torch.load(MODEL_DIR + 'distilbert_best.pt', map_location=device))
model.eval()

preds, targets = [], []
with torch.no_grad():
    for batch in loaders['test']:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds.extend(torch.argmax(outputs.logits, dim=1).cpu().tolist())
        targets.extend(batch['label'].tolist())

test_acc = accuracy_score(targets, preds)
report   = classification_report(targets, preds, target_names=label_names)
print(f'Test Accuracy : {test_acc:.4f}\n{report}')

cm = confusion_matrix(targets, preds)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title(f'Matrice de confusion — DistilBERT (acc={test_acc:.4f})')
plt.ylabel('Vrai'); plt.xlabel('Prédit')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(MODEL_DIR + 'distilbert_confusion.png', dpi=100)
plt.show()

## Bloc 8 : Sauvegarde des résultats

In [ ]:
results_distilbert = {
    'model': 'DistilBERT', 'test_accuracy': test_acc, 'best_val_acc': best_val_acc,
    'train_losses': train_losses, 'val_losses': val_losses, 'val_accs': val_accs,
    'predictions': preds, 'targets': targets, 'report': report,
}
with open(DATA_DIR + 'results_distilbert.pkl', 'wb') as f:
    pickle.dump(results_distilbert, f)
print(f'Résultats → {DATA_DIR}results_distilbert.pkl')
print(f'Modèle    → {MODEL_DIR}distilbert_best.pt')
print(f'Test accuracy : {test_acc:.4f}')